<a href="https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HeyOmkar07/flyrank-ml-internship-omkar/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
!pip -q install duckdb fsspec

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN is missing")

HF_TOKEN = HF_TOKEN.strip()

print("HF_TOKEN loaded successfully")

HF_TOKEN loaded successfully


In [3]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-02/data_0.parquet",
    repo_type="dataset"
)

print("Downloaded successfully:")
print(file_path)

Downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-02/data_0.parquet


In [4]:
import pandas as pd

df = pd.read_parquet(file_path)

print("Shape:", df.shape)

Shape: (7355108, 30)


In [5]:
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57.0,0.0,1778.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13.0,0.0,85.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59.0,0.0,1001.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17.0,0.0,287.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6.0,0.0,27.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

In [7]:
print(df.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [8]:
[c for c in df.columns if any(x in c.lower() for x in ["age", "date", "update", "refresh", "publish"])]

['report_date',
 'ga4_pageviews',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec']

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

1. Signal checks

In [9]:
import numpy as np
import pandas as pd

df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

df[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr"]].head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
0,57.0,0.0,31.192982,0.0
1,13.0,0.0,6.538462,0.0
2,59.0,0.0,16.966102,0.0
3,17.0,0.0,16.882353,0.0
4,6.0,0.0,4.500000,0.0


In [10]:
df["position_bucket"] = pd.cut(
    df["gsc_avg_position"],
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

In [11]:
position_ctr = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          avg_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

position_ctr

,position_bucket,n,avg_ctr,median_ctr
0,1-3,635987,0.004211,0.0
1,4-10,1121355,0.003496,0.0
2,11-20,420091,0.002474,0.0
3,21-50,336038,0.001783,0.0
4,51+,108311,0.000889,0.0


### Signal 1 — CTR vs Average Position

**Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**

CTR generally [increases/decreases/is inconsistent] as average position changes.
This [supports/does not support] the idea behind the CTR-fix logic.

### Signal 2 — Impression Volume

**Verdict: [CONFIRMED / OPPOSITE / MIXED / FALSE]**

The volume buckets show that [your observation].
This [supports/does not support] using impressions as a prioritization signal for quick-win opportunities.

In [12]:
df["volume_bucket"] = pd.cut(
    df["gsc_impressions"],
    bins=[0, 100, 500, 1000, 5000, float("inf")],
    labels=["0-100", "101-500", "501-1K", "1K-5K", "5K+"],
    include_lowest=True
)

In [13]:
volume_table = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("gsc_impressions", "size"),
          avg_impressions=("gsc_impressions", "mean"),
          avg_clicks=("gsc_clicks", "mean"),
          avg_ctr=("ctr", "mean")
      )
      .reset_index()
)

volume_table

,volume_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,0-100,6837161,6.557327,0.018621,0.003152
1,101-500,366372,207.142167,0.687258,0.003295
2,501-1K,42197,683.104178,2.308339,0.003383
3,1K-5K,16738,1641.794778,5.852491,0.003483
4,5K+,384,8070.369792,30.554688,0.004888


In [14]:
position_ctr

,position_bucket,n,avg_ctr,median_ctr
0,1-3,635987,0.004211,0.0
1,4-10,1121355,0.003496,0.0
2,11-20,420091,0.002474,0.0
3,21-50,336038,0.001783,0.0
4,51+,108311,0.000889,0.0


In [15]:
volume_table

,volume_bucket,n,avg_impressions,avg_clicks,avg_ctr
0,0-100,6837161,6.557327,0.018621,0.003152
1,101-500,366372,207.142167,0.687258,0.003295
2,501-1K,42197,683.104178,2.308339,0.003383
3,1K-5K,16738,1641.794778,5.852491,0.003483
4,5K+,384,8070.369792,30.554688,0.004888


In [16]:
df["ctr"]
df["position_bucket"]

,position_bucket
0,21-50
1,4-10
2,11-20
3,11-20
4,4-10
...,...
7355103,NaN
7355104,NaN
7355105,NaN
7355106,NaN


In [17]:
df[[
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "position_bucket"
]].head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,position_bucket
0,content_7995404695ee1ffd,57.0,0.0,31.192982,0.0,21-50
1,content_1eea820697c3b95a,13.0,0.0,6.538462,0.0,4-10
2,content_ccbb253f142217c3,59.0,0.0,16.966102,0.0,11-20
3,content_ae16a6b9cf64c80a,17.0,0.0,16.882353,0.0,11-20
4,content_acf700633f016e5a,6.0,0.0,4.500000,0.0,4-10


In [18]:
position_benchmark = (
    df.groupby("position_bucket", observed=False)["ctr"]
      .median()
      .rename("benchmark_ctr")
      .reset_index()
)

position_benchmark

,position_bucket,benchmark_ctr
0,1-3,0.0
1,4-10,0.0
2,11-20,0.0
3,21-50,0.0
4,51+,0.0


In [19]:
df = df.merge(
    position_benchmark,
    on="position_bucket",
    how="left"
)

In [20]:
df[[
    "gsc_avg_position",
    "ctr",
    "position_bucket",
    "benchmark_ctr"
]].head(10)

,gsc_avg_position,ctr,position_bucket,benchmark_ctr
0,31.192982,0.000000,21-50,0.0
1,6.538462,0.000000,4-10,0.0
2,16.966102,0.000000,11-20,0.0
3,16.882353,0.000000,11-20,0.0
4,4.500000,0.000000,4-10,0.0
5,6.925926,0.037037,4-10,0.0
6,24.600000,0.000000,21-50,0.0
7,18.200000,0.000000,11-20,0.0
8,31.740741,0.000000,21-50,0.0
9,3.291667,0.000000,4-10,0.0


In [21]:
df["ctr_gap"] = (
    df["benchmark_ctr"] - df["ctr"]
).clip(lower=0)

In [22]:
df["baseline_score"] = (
    df["ctr_gap"] * df["gsc_impressions"]
)

In [23]:
df.loc[
    ~df["gsc_avg_position"].between(1, 20),
    "baseline_score"
] = 0

In [24]:
df["reason_code"] = np.where(
    df["baseline_score"] > 0,
    "LOW_CTR_FOR_POSITION",
    "NO_OPPORTUNITY"
)

In [25]:
df["action"] = np.where(
    df["baseline_score"] > 0,
    "CTR_FIX",
    "NO_ACTION"
)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.